# 🌿 EcoQuest Species Classifier
### Fine-tunes EfficientNet-B0 on iNaturalist San Diego species data

**Outputs:**
- `ecoquest_classifier.pt` — trained model weights
- `ecoquest_labels.json` — index → species info mapping

**Steps:**
1. Install dependencies
2. Load & inspect the CSV
3. Build dataset (local files or URL fallback)
4. Fine-tune EfficientNet-B0 with 2-GPU support
5. Evaluate & save
6. Test inference
7. Lambda-ready inference function

## Cell 1 — Install dependencies

In [ ]:
!pip install torch torchvision Pillow scikit-learn pandas requests tqdm --quiet

## Cell 2 — Imports & config

In [ ]:
import os
import io
import json
import time
import base64
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# ── Config ────────────────────────────────────────────────────────────────────
CSV_PATH         = 'mostAbundant.csv'       # update path if needed
IMG_DIR          = 'cv_dataset'             # local image folder
MODEL_OUT        = 'ecoquest_classifier.pt'
LABELS_OUT       = 'ecoquest_labels.json'
BATCH_SIZE       = 64       # doubled for 2 GPUs
EPOCHS           = 10
LR               = 2e-4    # scaled up for larger batch
IMG_SIZE         = 224
USE_URL_FALLBACK = True

# ── Multi-GPU setup ───────────────────────────────────────────────────────────
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    DEVICE = torch.device('cuda')
    print(f'✅ {n_gpus} GPU(s) detected:')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'   GPU {i}: {props.name} ({props.total_memory // 1024**2} MB VRAM)')
else:
    n_gpus = 0
    DEVICE = torch.device('cpu')
    print('⚠️  No GPU detected — running on CPU')

print(f'\nUsing device: {DEVICE}')
print(f'Batch size:    {BATCH_SIZE}')
print(f'Learning rate: {LR}')
print(f'PyTorch:       {torch.__version__}')

## Cell 3 — Load CSV & build label map

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df['common_name'] != 'Unknown'].reset_index(drop=True)

print(f'Total images:  {len(df)}')
print(f'Total species: {df["scientific_name"].nunique()}')
print(f'\nTop 10 most represented species:')
print(df['scientific_name'].value_counts().head(10))
df.head(3)

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['scientific_name'])
n_classes   = len(le.classes_)
print(f'Number of classes: {n_classes}')

label_map = {}
for idx, sci_name in enumerate(le.classes_):
    row = df[df['scientific_name'] == sci_name].iloc[0]
    label_map[str(idx)] = {
        'scientific_name': sci_name,
        'common_name':     row['common_name'],
        'taxon_id':        int(row['taxon_id']),
    }

with open(LABELS_OUT, 'w') as f:
    json.dump(label_map, f, indent=2)

print(f'✅ Label map saved → {LABELS_OUT}')
print(f'\nSample entries:')
for k in list(label_map.keys())[:3]:
    print(f'  [{k}] {label_map[k]}')

## Cell 4 — Dataset class

In [ ]:
class SpeciesDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def load_image(self, idx):
        row      = self.df.iloc[idx]
        filepath = row['filepath']

        if os.path.exists(filepath):
            return Image.open(filepath).convert('RGB')

        if USE_URL_FALLBACK and pd.notna(row.get('image_url')):
            try:
                response = requests.get(row['image_url'], timeout=10)
                return Image.open(io.BytesIO(response.content)).convert('RGB')
            except Exception as e:
                print(f'  ⚠️  Failed: {row["image_url"]}: {e}')

        return Image.new('RGB', (IMG_SIZE, IMG_SIZE), color=(128, 128, 128))

    def __getitem__(self, idx):
        image = self.load_image(idx)
        label = int(self.df.iloc[idx]['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

print('✅ SpeciesDataset class defined')

## Cell 5 — Transforms & data splits

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

train_dataset = SpeciesDataset(train_df, transform=train_transform)
val_dataset   = SpeciesDataset(val_df,   transform=val_transform)

# num_workers=0 — safest for Jupyter/cloud environments
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=False
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
    persistent_workers=False
)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Cell 6 — Build model (EfficientNet-B0 + 2-GPU)

In [ ]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

in_features         = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, n_classes)

if n_gpus > 1:
    print(f'Wrapping model in DataParallel across {n_gpus} GPUs...')
    model = nn.DataParallel(model, device_ids=list(range(n_gpus)))

model = model.to(DEVICE)

base_model = model.module if isinstance(model, nn.DataParallel) else model
for name, param in base_model.named_parameters():
    if any(b in name for b in ['features.6', 'features.7', 'features.8', 'classifier']):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Output classes:       {n_classes}')
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
print(f'Multi-GPU:            {isinstance(model, nn.DataParallel)}')

## Cell 7 — Loss, optimizer, scheduler

In [ ]:
base_model = model.module if isinstance(model, nn.DataParallel) else model

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, base_model.parameters()),
    lr=LR, weight_decay=1e-4
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(f'✅ Optimizer: AdamW | LR: {LR} | Scheduler: CosineAnnealing')

## Cell 8 — Training loop with progress bars

In [ ]:
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

print(f'Starting training for {EPOCHS} epochs...')
print(f'GPUs active: {n_gpus if n_gpus > 1 else 1}')
print(f'Effective batch size: {BATCH_SIZE * max(n_gpus, 1)}')
print('=' * 80)

epoch_bar = tqdm(
    range(1, EPOCHS + 1),
    desc='Training', unit='epoch', colour='green', position=0, leave=True
)

for epoch in epoch_bar:

    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    t0 = time.time()

    train_bar = tqdm(
        train_loader,
        desc=f'  Epoch {epoch:02d}/{EPOCHS} Train',
        unit='batch', colour='blue', position=1, leave=False
    )

    for images, labels in train_bar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * images.size(0)
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += images.size(0)

        train_bar.set_postfix({
            'loss': f'{train_loss/train_total:.4f}',
            'acc':  f'{train_correct/train_total:.3f}',
        })

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    val_bar = tqdm(
        val_loader,
        desc=f'  Epoch {epoch:02d}/{EPOCHS} Val  ',
        unit='batch', colour='yellow', position=1, leave=False
    )

    with torch.no_grad():
        for images, labels in val_bar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs        = model(images)
            loss           = criterion(outputs, labels)

            val_loss    += loss.item() * images.size(0)
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += images.size(0)

            val_bar.set_postfix({
                'loss': f'{val_loss/val_total:.4f}',
                'acc':  f'{val_correct/val_total:.3f}',
            })

    train_acc = train_correct / train_total
    val_acc   = val_correct   / val_total
    elapsed   = time.time() - t0

    history['train_loss'].append(train_loss / train_total)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss / val_total)
    history['val_acc'].append(val_acc)

    scheduler.step()

    flag = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        base_model   = model.module if isinstance(model, nn.DataParallel) else model
        torch.save({
            'model_state_dict': base_model.state_dict(),
            'n_classes':        n_classes,
            'label_map':        label_map,
            'val_acc':          val_acc,
            'epoch':            epoch,
        }, MODEL_OUT)
        flag = ' ✅'

    vram_info = ''
    if n_gpus > 1:
        vram_parts = []
        for i in range(n_gpus):
            used      = torch.cuda.memory_allocated(i) // 1024**2
            total_mem = torch.cuda.get_device_properties(i).total_memory // 1024**2
            vram_parts.append(f'G{i}:{used}/{total_mem}MB')
        vram_info = ' | ' + ' '.join(vram_parts)

    epoch_bar.set_postfix({
        'train_acc': f'{train_acc:.3f}',
        'val_acc':   f'{val_acc:.3f}',
        'best':      f'{best_val_acc:.3f}',
        'time':      f'{elapsed:.1f}s',
    })

    tqdm.write(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'Train Loss: {train_loss/train_total:.4f} | Train Acc: {train_acc:.3f} | '
        f'Val Loss: {val_loss/val_total:.4f} | Val Acc: {val_acc:.3f} | '
        f'{elapsed:.1f}s{vram_info}{flag}'
    )

print('=' * 80)
print(f'\n🏆 Training complete!')
print(f'   Best val accuracy: {best_val_acc:.3f} ({best_val_acc*100:.1f}%)')
print(f'   Model saved → {MODEL_OUT}')

## Cell 9 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
ax1.plot(epochs_range, history['val_loss'],   label='Val Loss',   marker='o')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, history['train_acc'], label='Train Acc', marker='o')
ax2.plot(epochs_range, history['val_acc'],   label='Val Acc',   marker='o')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True)

plt.suptitle('EcoQuest Species Classifier — Training History', fontsize=14)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved → training_curves.png')

## Cell 10 — Test inference on a single image

In [ ]:
def predict_species(image_source, model_path=MODEL_OUT, top_k=3):
    checkpoint = torch.load(model_path, map_location='cpu')
    lmap       = checkpoint['label_map']
    nc         = checkpoint['n_classes']

    m = models.efficientnet_b0(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, nc)
    m.load_state_dict(checkpoint['model_state_dict'])
    m.eval()

    if isinstance(image_source, Image.Image):
        image = image_source.convert('RGB')
    elif isinstance(image_source, str) and image_source.startswith('http'):
        response = requests.get(image_source, timeout=10)
        image    = Image.open(io.BytesIO(response.content)).convert('RGB')
    elif isinstance(image_source, str) and os.path.exists(image_source):
        image = Image.open(image_source).convert('RGB')
    else:
        img_bytes = base64.b64decode(image_source)
        image     = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        outputs          = m(tensor)
        probs            = torch.softmax(outputs, dim=1)[0]
        top_probs, top_idxs = torch.topk(probs, k=min(top_k, nc))

    results = []
    for prob, idx in zip(top_probs.tolist(), top_idxs.tolist()):
        info = lmap[str(idx)]
        results.append({
            'common_name':     info['common_name'],
            'scientific_name': info['scientific_name'],
            'taxon_id':        info['taxon_id'],
            'confidence':      round(prob * 100, 1),
        })
    return results


sample_url  = df.iloc[0]['image_url']
print(f'Testing with: {sample_url}\n')
predictions = predict_species(sample_url)
print('Top predictions:')
for i, p in enumerate(predictions, 1):
    print(f'  {i}. {p["common_name"]} ({p["scientific_name"]}) — {p["confidence"]}%')

## Cell 11 — Test on 5 random validation images

In [ ]:
sample  = val_df.sample(5, random_state=99)
correct = 0

for _, row in sample.iterrows():
    preds  = predict_species(row['image_url'])
    top1   = preds[0]
    match  = top1['scientific_name'] == row['scientific_name']
    symbol = '✅' if match else '❌'
    if match:
        correct += 1
    print(f'{symbol} True: {row["common_name"]:30s} | Pred: {top1["common_name"]:30s} | Conf: {top1["confidence"]}%')

print(f'\nSample accuracy: {correct}/5')

## Cell 12 — Lambda inference code (for backend team)
Paste this into your `POST /species/identify` Lambda function.

In [ ]:
LAMBDA_CODE = '''
import json, base64, io, boto3, torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image

s3         = boto3.client('s3')
BUCKET     = 'your-ecoquest-bucket'
MODEL_KEY  = 'models/ecoquest_classifier.pt'
LABELS_KEY = 'models/ecoquest_labels.json'

def load_model():
    obj        = s3.get_object(Bucket=BUCKET, Key=MODEL_KEY)
    checkpoint = torch.load(io.BytesIO(obj['Body'].read()), map_location='cpu')
    model = models.efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, checkpoint['n_classes'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    obj       = s3.get_object(Bucket=BUCKET, Key=LABELS_KEY)
    label_map = json.loads(obj['Body'].read())
    return model, label_map

model, label_map = load_model()

TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

CORS = {
    'Access-Control-Allow-Origin':  '*',
    'Access-Control-Allow-Headers': 'Content-Type',
    'Access-Control-Allow-Methods': 'POST,OPTIONS',
    'Content-Type':                 'application/json',
}

def lambda_handler(event, context):
    if event.get('httpMethod') == 'OPTIONS':
        return {'statusCode': 200, 'headers': CORS, 'body': ''}
    try:
        body      = json.loads(event['body'])
        img_bytes = base64.b64decode(body['image'])
        image     = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        tensor    = TRANSFORM(image).unsqueeze(0)
        with torch.no_grad():
            outputs    = model(tensor)
            probs      = torch.softmax(outputs, dim=1)[0]
            confidence = probs.max().item()
            pred_idx   = probs.argmax().item()
        species = label_map[str(pred_idx)]
        return {
            'statusCode': 200,
            'headers':    CORS,
            'body':       json.dumps({
                'common_name':     species['common_name'],
                'scientific_name': species['scientific_name'],
                'taxon_id':        species['taxon_id'],
                'confidence':      round(confidence * 100, 1),
            })
        }
    except Exception as e:
        return {'statusCode': 500, 'headers': CORS, 'body': json.dumps({'error': str(e)})}
'''

with open('lambda_identify.py', 'w') as f:
    f.write(LAMBDA_CODE)

print('✅ Lambda code saved → lambda_identify.py')
print()
print('Next steps for backend team:')
print('  1. Upload ecoquest_classifier.pt  → S3')
print('  2. Upload ecoquest_labels.json    → S3')
print('  3. Paste lambda_identify.py into POST /species/identify Lambda')
print('  4. Set BUCKET to your actual S3 bucket name')
print('  5. Add torch + torchvision + Pillow as Lambda layers')

## Cell 13 — Upload model files to S3 (uncomment when ready)

In [ ]:
# import boto3
# BUCKET = 'your-ecoquest-bucket'
# s3     = boto3.client('s3')
# s3.upload_file(MODEL_OUT,  BUCKET, f'models/{MODEL_OUT}')
# print(f'✅ Uploaded {MODEL_OUT}  → s3://{BUCKET}/models/{MODEL_OUT}')
# s3.upload_file(LABELS_OUT, BUCKET, f'models/{LABELS_OUT}')
# print(f'✅ Uploaded {LABELS_OUT} → s3://{BUCKET}/models/{LABELS_OUT}')

print('Uncomment the code above and set your bucket name to upload to S3')